In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, explode
from delta.tables import DeltaTable
import pandas as pd

In [0]:
BASE_PATH = "/Volumes/workspace/project_data_football_raw/partidas_raw"

In [0]:
# Função para obter última rodada processada
def get_ultima_rodada_processada():
    try:
        result = spark.sql("""
            SELECT MAX(rodada) as ultima
            FROM project_data_football_bronze.partidas_rodada
        """).collect()[0]["ultima"]

        return result if result is not None else 0
    except:
        return 0

In [0]:
# Obtém a última rodada processada
ultima_processada = get_ultima_rodada_processada()

print("Última rodada processada:", ultima_processada)

# Lista os diretórios das rodadas no caminho base
rodadas_dirs = dbutils.fs.ls(BASE_PATH)

In [0]:
for r in rodadas_dirs:

    # Verifica se o diretório corresponde ao padrão esperado de rodada
    if not r.name.startswith("rodada="):
        continue

    # Extrai o número da rodada do nome do diretório
    rodada = int(r.name.replace("rodada=", "").replace("/", ""))

    print(f"Iniciando ingestão da rodada {rodada}")

    # Lista os arquivos dentro do diretório da rodada
    arquivos_na_pasta = dbutils.fs.ls(r.path)
    arquivos_ordenados = sorted(arquivos_na_pasta, key=lambda x: x.name, reverse=True)

    if not arquivos_ordenados:
        continue

    # PEGA O PATH (STRING) DO ARQUIVO
    ultimo_arquivo_path = arquivos_ordenados[0].path
    print(f"Lendo arquivo: {ultimo_arquivo_path}")

    # 2. LEITURA CORRETA
    df_raw = spark.read.json(ultimo_arquivo_path)
    
    # 3. EXPLODE (Conforme você já fez, está certinho)
    from pyspark.sql.functions import explode
    df_partidas = df_raw.select(explode("partidas").alias("p")).select("p.*")

    if df_partidas.count() == 0:
        print(f"Nenhuma partida encontrada para rodada {rodada}")
        continue

    # 2. FILTRO DE QUALIDADE
    df_partidas = df_partidas.filter("placar_oficial_mandante IS NOT NULL")

    # 3. AJUSTE DE TIPOS E METADADOS
    df_partidas = (
        df_partidas
        .withColumn("placar_oficial_mandante", col("placar_oficial_mandante").cast("int"))
        .withColumn("placar_oficial_visitante", col("placar_oficial_visitante").cast("int"))
        .withColumn("rodada", lit(rodada))
        .withColumn("dt_ingestao", current_timestamp())
    )

    tabela = "project_data_football_bronze.partidas_rodada"

    # Verifica se a tabela Delta já existe
    if spark.catalog.tableExists(tabela):

        # Realiza merge das partidas na tabela Delta
        delta_table = DeltaTable.forName(spark, tabela)

        (
            delta_table.alias("target") \
            .merge(
                df_partidas.alias("source"),
                "target.partida_id = source.partida_id"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        )

        print(f"Rodada {rodada} mesclada com sucesso")

    else:

        # Cria a tabela Delta caso não exista e salva as partidas
        df_partidas.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(tabela)

        print(f"Tabela criada e rodada {rodada} carregada")

print("Ingestão finalizada com sucesso.")